## Initialization

In [ ]:
# Imports
# import pickle
# from pathlib import Path
from typing import Callable, Literal
from random import sample

import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
from scipy.signal import find_peaks
from scipy.interpolate import CubicSpline
import pandas as pd
# from scipy.interpolate import interp1d
# from data_processing.processing.bimodal_fitting import (
#     get_psd_energy_histogram,
#     scan_histogram_slices,
#     find_failed_slices,
#     BimodalBounds,
#     BimodalParams
# )
# from data_processing.processing.calibration import Detector, recalibrate
# from data_processing.processing.figure_of_merit import gaussian
# from data_processing.processing.neutron_classification import classify
# from data_processing.processing.neutron_window_generation import (
#     generate_nasa_neutron_window,
#     generate_n_distro_neutron_window
# )
from data_processing import processing as proc
from data_processing import loading as load
from data_processing import types as proc_types
# from data_processing.arc_paths import (INPUT_DATA_FOLDER, get_exp_root,
#                                        get_parq_root)
from data_processing.dataframe_validation import DetectorDataframeColumn, EnergyColumn
from data_processing.experiment_data_keys import (ExperimentDataKey,
                                                  ExperimentNeutronData)
from data_processing.helpers import (
    get_input_with_default,
    # get_input_required,
    input_experiment_ids,
    stop
)
# from data_processing.loading import get_neutron_window_paths, load_side_borders
# from data_processing.loading.dataframe_loading import load_psd
# from data_processing.loading.timetag_processing import calculate_timetag_hours
# from data_processing.reporting import plot_classification
# from data_processing.types import (BimodalBounds, BimodalParams,
#                                    NasaGenerationSettings,
#                                    NeutronWindowSettings, WindowType)
# from scipy.optimize import curve_fit
# from scipy.signal import deconvolve

In [ ]:
CalibrationKey = Literal[ExperimentDataKey.CAEN_CALIBRATION, ExperimentDataKey.NEW_CALIBRATION]
NasaBorderKey = Literal[ExperimentDataKey.NASA_BORDERS, ExperimentDataKey.NASA_BORDERS_RECALC]

### Functions

In [ ]:
def get_nasa_generation_settings(
    calib_key: CalibrationKey
) -> proc_types.NasaGenerationSettings:
    sigma = get_input_with_default(
        """\
Enter value of sigma
Press Enter for default (5)
""",
        5,
        float
    )
    window_offset = get_input_with_default(
        """\
Enter value of offset between top and bottom window border
Press Enter for default (0.2)
""",
        0.2,
        float
    )
    left_border_type_input = get_input_with_default(
        """\
How do you want to handle the left border?
1: use existing value (default)
2: recalculate from data
3: enter own value
Press Enter for default
""",
        1,
        int
    )
    if left_border_type_input == 1:
        existing_left_border_version_input = get_input_with_default(
            """\
Which existing left border do you want to use?
1: original (0.1966 MeVee) (default)
2: newer (~0.1866 MeVee)
or press Enter for default
""",
            1,
            int
        )
        if existing_left_border_version_input in [1, 2]:
            border_key = ExperimentDataKey.NASA_BORDERS if existing_left_border_version_input == 1 else ExperimentDataKey.NASA_BORDERS_RECALC
            file_name_prefix = f"{calib_key.value}_{border_key.value}"
            side_borders_path, *_ = load.get_neutron_window_paths(file_name_prefix=file_name_prefix)
            left_border, _ = load.load_side_borders(side_borders_path=side_borders_path)
            if left_border is None:
                raise ValueError("Left border could not be loaded")
            lower_energy_bound = left_border
            recalc_lower_bound = False
        else:
            raise ValueError("Unsupported choice")
        pass
    elif left_border_type_input == 2:
        lower_energy_bound = 0.1966
        recalc_lower_bound = True
    elif left_border_type_input == 3:
        lower_energy_bound = get_input_with_default(
            """\
Enter value of lower energy bound (in MeVee)
Press Enter for default (0.1966)
""",
            0.1966,
            float
        )
        recalc_lower_bound = False
    else:
        raise ValueError("Unsupported choice")
    settings = proc_types.NasaGenerationSettings(
        window_offset=window_offset,
        sigma=sigma,
        lower_energy_bound=lower_energy_bound,
        recalculate_lower_energy_bound=recalc_lower_bound
    )
    return settings


def make_strategy_factory_fn(
    strategy_factory: proc.NeutronStrategyFactory,
    window_type: proc_types.WindowType,
    loading: bool,
    settings: proc_types.NeutronWindowSettings
) -> Callable[[], proc.AbstractNeutronStrategy]:
    def factory_fn():
        return strategy_factory.make_neutron_window_strategy(
            window_type, loading, settings
        )
    return factory_fn


def make_strategy_for_experiments(
    experiment_neutron_data: ExperimentNeutronData, 
    factory_fn: Callable[[], proc.AbstractNeutronStrategy]
) -> ExperimentNeutronData:
    new_neutron_data = {
        exp_id: {**exp_data, ExperimentDataKey.BORDER_STRATEGY: factory_fn()}
        for exp_id, exp_data
        in experiment_neutron_data.items()
    }
    return new_neutron_data

In [ ]:
def moving_average(arr, n=5):
    ret = np.cumsum(arr, dtype=float)
    ret[n:] = ret[n:] - ret[:-n]
    mov_avg = ret[n-1:] / n
    prefix = np.empty((n-1,))
    prefix[:] = np.nan
    return np.concatenate((prefix, mov_avg))


def moving_average_centered(arr, n=5):
    if n % 2 != 1:
        raise ValueError("Centered moving average needs odd window size")
    prefix_count = (n-1)//2
    ret = np.nancumsum(arr, dtype=float)
    ret[n:] = ret[n:] - ret[:-n]
    mov_avg = ret[n-1:] / n
    prefix = np.empty((prefix_count,))
    suffix = np.empty((prefix_count,))
    prefix[:] = np.nan
    suffix[:] = np.nan
    return np.concatenate((prefix, mov_avg, suffix))

## Experiment ID Input

In [ ]:
# experiment_ids = input_experiment_ids()
voltages = [1250, 1300, 1350, 1400, 1450, 1475, 1500, 1525, 1550]
# experiment_ids = ["TB-unmatched", "TB-matched"]
experiment_ids = [f"TB-bias-{voltage}" for voltage in voltages]

In [ ]:
# more here?
experiment_neutron_data: ExperimentNeutronData = {
    exp_id: {}
    for exp_id in experiment_ids
}

In [ ]:
# calib_input = get_input_with_default(
#     "Do you want to use new calibration? [y/n, or press Enter for yes]",
#     "y",
#     str
# )
calib_input = "y"

is_new_calibration = calib_input.lower() == "y"
calibrated_energy_column: EnergyColumn = (
    DetectorDataframeColumn.RECALIBRATED_ENERGY
    if is_new_calibration else DetectorDataframeColumn.CALIB_ENERGY
)
calib_key: CalibrationKey = ExperimentDataKey.NEW_CALIBRATION if is_new_calibration else ExperimentDataKey.CAEN_CALIBRATION

In [ ]:
strategy_factory = proc.NeutronStrategyFactory()
# settings = get_nasa_generation_settings(calib_key)
window_offset = 0.2
sigma = 5
lower_energy_bound = 0.05
recalc_lower_bound = False
settings = proc_types.NasaGenerationSettings(
        window_offset=window_offset,
        sigma=sigma,
        lower_energy_bound=lower_energy_bound,
        recalculate_lower_energy_bound=recalc_lower_bound
    )
factory_fn = make_strategy_factory_fn(
    strategy_factory, "nasa", False, settings)
experiment_neutron_data = make_strategy_for_experiments(
    experiment_neutron_data, factory_fn)

## Data Loading and Initial Processing

In [ ]:
# Data Loading
for exp_id, exp_data in experiment_neutron_data.items():
    exp_data[ExperimentDataKey.UNCLASSIFIED] = load.load_caen_csvs(exp_id, raw=True)
    exp_data["signals_df"] = load.load_caen_csvs(exp_id, get_psd=False, get_signals=True, raw=True)

In [ ]:
# Express timetags in hours elapsed
for exp_id, exp_data in experiment_neutron_data.items():
    unclassified_df = exp_data[ExperimentDataKey.UNCLASSIFIED]
    unclassified_df = load.calculate_timetag_hours(unclassified_df)
    exp_data[ExperimentDataKey.UNCLASSIFIED] = unclassified_df

In [ ]:
# Recalibrate energy
for exp_id, exp_data in experiment_neutron_data.items():
    unclassified_df = exp_data[ExperimentDataKey.UNCLASSIFIED]
    unclassified_df = proc.recalibrate(unclassified_df, proc.Detector.ZERO)
    exp_data[ExperimentDataKey.UNCLASSIFIED] = unclassified_df

## Neutron Classification

In [ ]:
# Generate histogram

start_scan_idx = 0
end_scan_idx = 420
energy_width = 20e-3

for exp_id, exp_data in experiment_neutron_data.items():
    psd_report = exp_data[ExperimentDataKey.UNCLASSIFIED]
    Z, xe, ye = proc.get_psd_energy_histogram(
        psd_report,
        calibrated_energy_column,
        energy_width=energy_width
    )
    exp_data[ExperimentDataKey.PSD_HISTOGRAM] = Z
    exp_data[ExperimentDataKey.HISTOGRAM_X_EDGES] = xe
    exp_data[ExperimentDataKey.HISTOGRAM_Y_EDGES] = ye
    exp_data[ExperimentDataKey.END_SCAN_IDX] = min(end_scan_idx, len(Z))

In [ ]:
# TODO get fit dataframe (not needed if loading, but do anyway to keep process consistent)
stop_here = False

for exp_id, exp_data in experiment_neutron_data.items():
    Z = exp_data[ExperimentDataKey.PSD_HISTOGRAM]
    xe = exp_data[ExperimentDataKey.HISTOGRAM_X_EDGES]
    ye = exp_data[ExperimentDataKey.HISTOGRAM_Y_EDGES]
    end_scan_idx = exp_data[ExperimentDataKey.END_SCAN_IDX]

    # # Default
    # default_bounds: BimodalBounds = (
    #     BimodalParams(0.1, 0.01, 1, 0.25, 0.01, 0),
    #     BimodalParams(0.2, 0.1, Z.max(), 0.38, 0.04, 4000)
    # )

    # bounds_a: BimodalBounds = (
    #     BimodalParams(0.1, 0.01, 1, 0.34, 0.01, 0),
    #     BimodalParams(0.2, 0.1, Z.max(), 0.36, 0.04, 4000)
    # )

    # bounds_b: BimodalBounds = (
    #     BimodalParams(0.1, 0.01, 1, 0.34, 0.01, 0),
    #     BimodalParams(0.2, 0.1, Z.max(), 0.36, 0.03, 4000)
    # )

    # # Ranged Example
    # bounds = [
    #     ((0, 60), bounds_a),
    # ]

    df, df_err = proc.scan_histogram_slices(
        Z,
        xe,
        ye,
        fit_style="peak_finder",
        # default_bounds,
        # bounds=bounds,
        start_idx=start_scan_idx,
        end_idx=end_scan_idx
    )
    df, bad_slice_indexes = proc.find_failed_slices(df, exp_id)

    if bad_slice_indexes is not None:
        exp_data[ExperimentDataKey.VALID_SLICE_FITS] = df
        exp_data[ExperimentDataKey.BAD_SLICE_INDEXES] = bad_slice_indexes
        stop_here = True
    else:
        # exp_data['fom_results'] = df
        exp_data[ExperimentDataKey.FOM_RESULTS] = df

if stop_here:
    stop()

In [ ]:
# get borders from strategy
for exp_id, exp_data in experiment_neutron_data.items():
    if ExperimentDataKey.FOM_RESULTS not in exp_data:
        print(f"No good fit data on Experiment {exp_id}")
        continue

    fom_results = exp_data[ExperimentDataKey.FOM_RESULTS]
    strategy = exp_data[ExperimentDataKey.BORDER_STRATEGY]

    strategy.set_slice_fit_dataframe(fom_results)
    borders = strategy.get_neutron_window()

    exp_data[ExperimentDataKey.BORDERS] = borders

In [ ]:
# classify neutrons
for exp_id, exp_data in experiment_neutron_data.items():
    psd_report = exp_data[ExperimentDataKey.UNCLASSIFIED].copy()
    borders = exp_data[ExperimentDataKey.BORDERS]

    psd_report = proc.classify(
        psd_report,
        calibrated_energy_column,
        borders,
        DetectorDataframeColumn.NEW_N_CLASS
    )

    exp_data[ExperimentDataKey.PSD_REPORT] = psd_report

## Pulse Selection

In [ ]:
signals_count = 1500000
for exp_id, exp_data in experiment_neutron_data.items():
    psd_report = exp_data[ExperimentDataKey.PSD_REPORT]
    n_class_col_name = DetectorDataframeColumn.NEW_N_CLASS.value

    print(exp_id)
    
    # gamma_only = psd_report.query(f"~{n_class_col_name}").copy()
    # neutrons_only = psd_report.query(n_class_col_name).copy()
    gamma_only = psd_report.query(f"~{n_class_col_name}").iloc[:signals_count, :].copy()
    neutrons_only = psd_report.query(n_class_col_name).iloc[:signals_count, :].copy()
    all_particles_subset = psd_report.iloc[:signals_count, :].copy()
    print(psd_report.shape)
    print(neutrons_only.shape)
    print(gamma_only.shape)
    print(all_particles_subset.shape)
    exp_data[ExperimentDataKey.NEUTRONS_ONLY] = neutrons_only
    exp_data[ExperimentDataKey.GAMMA_ONLY] = gamma_only
    exp_data["all_particles_subset"] = all_particles_subset

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    all_particles_subset = exp_data["all_particles_subset"]
    neutrons_only = exp_data[ExperimentDataKey.NEUTRONS_ONLY]
    gamma_only = exp_data[ExperimentDataKey.GAMMA_ONLY]
    signals_df = exp_data["signals_df"]
    
    neutron_signals = signals_df.loc[neutrons_only.index].astype("int32")
    # neutron_signals = signals_df.astype("int32")
    gamma_signals = signals_df.loc[gamma_only.index].astype("int32")
    subset_signals = signals_df.loc[all_particles_subset.index].astype("int32")

    n_signals_np = neutron_signals.to_numpy()
    # print(n_signals_np.shape)
    # n_baselines = n_signals_np.max(axis=1).reshape(-1, 1)
    n_baselines = n_signals_np[:, :30].mean(axis=1).reshape(-1, 1)
    n_signals_np = -n_signals_np + n_baselines
    # print(n_signals_np.max())
    neutron_signals = pd.DataFrame(n_signals_np, index=neutron_signals.index, columns=neutron_signals.columns)
    neutrons_only["peak_height"] = neutron_signals.max(axis=1)

    g_signals_np = gamma_signals.to_numpy()
    # g_baselines = g_signals_np.max(axis=1).reshape(-1, 1)
    g_baselines = g_signals_np[:, :30].mean(axis=1).reshape(-1, 1)
    g_signals_np = -g_signals_np + g_baselines
    # print(g_signals_np.max())
    gamma_signals = pd.DataFrame(g_signals_np, index=gamma_signals.index, columns=gamma_signals.columns)
    gamma_only["peak_height"] = gamma_signals.max(axis=1)

    sub_signals_np = subset_signals.to_numpy()
    # g_baselines = g_signals_np.max(axis=1).reshape(-1, 1)
    sub_baselines = sub_signals_np[:, :30].mean(axis=1).reshape(-1, 1)
    sub_signals_np = -sub_signals_np + sub_baselines
    # print(sub_signals_np.max())
    subset_signals = pd.DataFrame(sub_signals_np, index=subset_signals.index, columns=subset_signals.columns)
    all_particles_subset["peak_height"] = subset_signals.max(axis=1)

    exp_data["neutron_signals"] = neutron_signals
    exp_data["gamma_signals"] = gamma_signals
    exp_data["subset_signals"] = subset_signals

In [ ]:
height_bin_width = 100
energy_bin_width = 50
for exp_id, exp_data in experiment_neutron_data.items():
    # print(exp_id)
    neutrons_only = exp_data[ExperimentDataKey.NEUTRONS_ONLY]
    n_energy = neutrons_only["ENERGY"]
    n_peak_height = neutrons_only["peak_height"]
    gamma_only = exp_data[ExperimentDataKey.GAMMA_ONLY]
    g_energy = gamma_only["ENERGY"]
    g_peak_height = gamma_only["peak_height"]
    all_particles_subset = exp_data["all_particles_subset"]
    sub_energy = all_particles_subset["ENERGY"]
    sub_peak_height = all_particles_subset["peak_height"]
    # print(neutron_energies.max())
    # print(neutron_energies.min())

    height_bins = np.arange(0, 15000, step=height_bin_width)
    energy_bins = np.arange(0, 4500, step=energy_bin_width)
    
    Zh_n, *_ = np.histogram(n_peak_height, bins=height_bins)
    Zh_g, *_ = np.histogram(g_peak_height, bins=height_bins)
    Zh_sub, *_ = np.histogram(sub_peak_height, bins=height_bins)
    Ze_n, *_ = np.histogram(n_energy, bins=energy_bins)
    Ze_g, *_ = np.histogram(g_energy, bins=energy_bins)
    Ze_sub, *_ = np.histogram(sub_energy, bins=energy_bins)
    exp_data[ExperimentDataKey.PULSE_HEIGHT_DISTRIBUTION] = {
        "neutron": {"height": Zh_n, "height_bins": height_bins, "energy": Ze_n, "energy_bins": energy_bins},
        "gamma": {"height": Zh_g, "height_bins": height_bins, "energy": Ze_g, "energy_bins": energy_bins},
        "subset": {"height": Zh_sub, "height_bins": height_bins, "energy": Ze_sub, "energy_bins": energy_bins}
    }

In [ ]:
# moving average
for exp_id, exp_data in experiment_neutron_data.items():
    phd_histogram_data = exp_data[ExperimentDataKey.PULSE_HEIGHT_DISTRIBUTION]
    phd_nh_histogram = phd_histogram_data["neutron"]["height"]
    phd_gh_histogram = phd_histogram_data["gamma"]["height"]
    phd_subh_histogram = phd_histogram_data["subset"]["height"]
    phd_ne_histogram = phd_histogram_data["neutron"]["energy"]
    phd_ge_histogram = phd_histogram_data["gamma"]["energy"]
    phd_sube_histogram = phd_histogram_data["subset"]["energy"]

    phd_nh_moving_average = moving_average_centered(phd_nh_histogram)
    phd_gh_moving_average = moving_average_centered(phd_gh_histogram)
    phd_subh_moving_average = moving_average_centered(phd_subh_histogram)
    # TODO add subset
    phd_ne_moving_average = moving_average_centered(phd_ne_histogram)
    phd_ge_moving_average = moving_average_centered(phd_ge_histogram)
    phd_sube_moving_average = moving_average_centered(phd_sube_histogram)

    phd_histogram_data["neutron"]["height_moving_average"] = phd_nh_moving_average
    phd_histogram_data["gamma"]["height_moving_average"] = phd_gh_moving_average
    phd_histogram_data["subset"]["height_moving_average"] = phd_subh_moving_average
    phd_histogram_data["neutron"]["energy_moving_average"] = phd_ne_moving_average
    phd_histogram_data["gamma"]["energy_moving_average"] = phd_ge_moving_average
    phd_histogram_data["subset"]["energy_moving_average"] = phd_sube_moving_average
    exp_data[ExperimentDataKey.PULSE_HEIGHT_DISTRIBUTION] = phd_histogram_data

In [ ]:
max_phd_count = 0
for exp_id, exp_data in experiment_neutron_data.items():
    print(exp_id)
    phd_n_data = exp_data[ExperimentDataKey.PULSE_HEIGHT_DISTRIBUTION]["neutron"]
    phd_n_histogram = phd_n_data["height"]

    current_max = phd_n_histogram.max()
    if current_max > max_phd_count:
        max_phd_count = current_max
    print(current_max)
print(f"Max height: {max_phd_count}")

for exp_id, exp_data in experiment_neutron_data.items():
    print(exp_id)
    phd_n_data = exp_data[ExperimentDataKey.PULSE_HEIGHT_DISTRIBUTION]["subset"]
    phd_n_histogram = phd_n_data["height"]

    current_max = phd_n_histogram.max()
    h_norm_factor = 1 / current_max
    phd_n_data["h_norm_factor"] = h_norm_factor
    print(f"Norm factor: {h_norm_factor}")

In [ ]:
max_phd_count = 0
for exp_id, exp_data in experiment_neutron_data.items():
    print(exp_id)
    phd_n_data = exp_data[ExperimentDataKey.PULSE_HEIGHT_DISTRIBUTION]["subset"]
    phd_n_histogram = phd_n_data["energy"]

    current_max = phd_n_histogram.max()
    if current_max > max_phd_count:
        max_phd_count = current_max
    print(current_max)
print(f"Max energy: {max_phd_count}")

for exp_id, exp_data in experiment_neutron_data.items():
    print(exp_id)
    phd_n_data = exp_data[ExperimentDataKey.PULSE_HEIGHT_DISTRIBUTION]["subset"]
    phd_n_histogram = phd_n_data["energy"]

    current_max = phd_n_histogram.max()
    norm_factor = max_phd_count / current_max
    phd_n_data["norm_factor"] = norm_factor
    print(f"Norm factor: {norm_factor}")

## Plotting

In [ ]:
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial'] + plt.rcParams['font.sans-serif']
fontsize = 20

In [ ]:
bg_blue = "#4c94ff"
bg_red = "#f54336"
bg_grey = "#9e9e9e"
bg_bluegrey = "#8a9fb8"
bg_green = "#21d894"

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
fig.subplots_adjust(wspace=0.2)

annot_positions = {
    "1200": (1300, 1.0),
    "1250": (1350, 0.95),
    "1300": (1950, 0.59),
    "1350": (2750, 0.36),
    "1400": (4230, 0.216),
    "1450": (6350, 0.135),
    "1475": (7775, 0.109),
    "1500": (9425, 0.1),
    "1525": (11400, 0.08),
    "1550": (13850, 0.075)
}

for exp_id, exp_data in experiment_neutron_data.items():
    color = bg_blue if "1525" in exp_id else bg_grey
    alpha = 0.3 if "1525" in exp_id else 0.4
    # alpha = 0.3
    
    neutrons_only = exp_data[ExperimentDataKey.NEUTRONS_ONLY]
    phd_histogram_data = exp_data[
        ExperimentDataKey.PULSE_HEIGHT_DISTRIBUTION]["subset"]
    energy_bins = phd_histogram_data["height_bins"]
    phd_n_histogram = phd_histogram_data["height"]
    norm_factor = phd_histogram_data["h_norm_factor"]

    energy_bin_mids = (energy_bins[1:] + energy_bins[:-1]) / 2
    
    # ax.plot(energy_bin_mids, phd_n_histogram * norm_factor, label=exp_id, lw=0)
    ax.fill_between(energy_bin_mids, phd_n_histogram * norm_factor, color=color, alpha=alpha)
    ax.tick_params(labelsize=fontsize)
    ax.set_xlabel("Pulse height (ADC channel)", fontsize=fontsize)
    ax.set_ylabel("Normalized counts", fontsize=fontsize)
    voltage = exp_id[-4:]
    annot_pos = annot_positions[voltage]
    ax.annotate(f"{voltage} V", annot_pos, ha="center", va="baseline", fontsize=fontsize-2)

In [ ]:
input("Processing done, hit Enter to finish")
stop()